# Gemini 3.1 Pro — Full Corpus Analysis (280 videos)

Runs the winning prompt (Variant B, selected in pilot) against all 280 videos, streams results to a JSONL log in Drive, and emits a final CSV that pastes straight into the master spreadsheet.

## Things to Remember
- Discovers every `.mp4` under each generator folder — no hardcoded filenames
- Analyses each video with Variant B (the pilot winner)
- Retries JSON parse failures up to 2 times
- Checkpoints after every video (crash-safe — rerun the cell and it picks up where it left off)
- Tracks running cost estimate
- Emits a CSV in the exact column layout of your master spreadsheet


## 1. Install and import

In [1]:
!pip install -q google-genai
import os, re, json, time, pathlib, csv
from datetime import datetime
from collections import Counter
from google import genai
from google.genai import types
from google.colab import userdata, drive


## 2. Auth and Drive mount

In [2]:
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
drive.mount('/content/drive')
print("Auth OK. Drive mounted.")


Mounted at /content/drive
Auth OK. Drive mounted.


## 3. Config

Change `CORPUS_ROOT` if your shortcut lives somewhere other than `/content/drive/MyDrive/msc-deepfake/generated_videos_final`.


In [3]:
CORPUS_ROOT = "/content/drive/MyDrive/msc-deepfake/generated_videos_final"

GENERATOR_FOLDERS = {
    "LTX":         "ltx",
    "Hunyuan":     "hunyuan",
    "Wan":         "wan",
    "Kling":       "kling",
    "Gemini_Omni": "gemini_omni_flash",
    "Seedance":    "seedance",
    "Pexels":      "pexels",
}

EXPECTED_COUNTS = {
    "LTX": 40, "Hunyuan": 40, "Wan": 40,
    "Kling": 32, "Gemini_Omni": 32, "Seedance": 32,
    "Pexels": 64,
}

MODEL       = "gemini-3.1-pro-preview"
TEMPERATURE = 0.0
MAX_RETRIES = 2         # per-video parse retries
SLEEP_S     = 1         # gentle rate-limit buffer between calls

OUT_DIR = pathlib.Path("/content/drive/MyDrive/msc-deepfake/gemini_full_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "gemini_results.jsonl"     # append-only log
CSV_PATH = OUT_DIR / "gemini_results.csv"       # final tabular output
print(f"Log: {LOG_PATH}\nCSV: {CSV_PATH}")


Log: /content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.jsonl
CSV: /content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.csv


## 4. Discover videos and sanity-check counts

In [4]:
root = pathlib.Path(CORPUS_ROOT)
assert root.exists(), (
    f"CORPUS_ROOT not found: {root}\n"
    f"Have you added the Shared-with-me folder to your Drive as a shortcut?"
)

VIDEO_INDEX = []   # list of (generator_label, absolute_path)
for gen_label, subdir in GENERATOR_FOLDERS.items():
    folder = root / subdir
    if not folder.exists():
        print(f"  WARNING: {folder} does not exist — skipping {gen_label}")
        continue
    files = sorted(folder.glob("*.mp4"))
    expected = EXPECTED_COUNTS.get(gen_label, "?")
    marker = "✓" if len(files) == expected else "!"
    print(f"  {marker} {gen_label:14s} found {len(files):3d} / expected {expected}   ({folder})")
    for f in files:
        VIDEO_INDEX.append((gen_label, str(f)))

print(f"\nTotal videos to process: {len(VIDEO_INDEX)}")


  ✓ LTX            found  40 / expected 40   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/ltx)
  ✓ Hunyuan        found  40 / expected 40   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/hunyuan)
  ✓ Wan            found  40 / expected 40   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/wan)
  ✓ Kling          found  32 / expected 32   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/kling)
  ✓ Gemini_Omni    found  32 / expected 32   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/gemini_omni_flash)
  ✓ Seedance       found  32 / expected 32   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/seedance)
  ✓ Pexels         found  64 / expected 64   (/content/drive/MyDrive/msc-deepfake/generated_videos_final/pexels)

Total videos to process: 280


## 5. The prompt (Variant B)

In [5]:
SYSTEM_PROMPT = "You are a forensic examiner analysing a short video clip for artefacts characteristic of AI-generated content (text-to-video models). Identify which artefact families, if any, are present, using the taxonomy below. The video may or may not be AI-generated; base your judgment on evidence in the video itself.\n\n# ARTEFACT TAXONOMY (12 families)\n\n## Domain 1: Surface Artefacts (per-frame visual failures)\n1.1 Texture defects \u2014 waxy skin, plastic-looking materials, repetitive backgrounds, loss of fine detail (fabric weave, hair strands, wood grain).\n1.2 Boundary defects \u2014 blurred or soft object edges, halo/bleed around subjects, chromatic fringing, subject-background dissolution.\n1.3 Lighting inconsistency \u2014 missing or wrong-direction shadows, impossible light sources, non-physical reflections, lighting mismatched to scene.\n1.4 Watermark / provenance signal \u2014 visible watermarks, logos, or overlay text indicating generator origin (e.g. \"Sora\", \"Veo\", \"Kling\"), OR conspicuous absence-of-noise patterns and frequency-domain regularities suggestive of synthesis. Judge only from what is visible in the pixels; do not infer from filename or context.\n\n## Domain 2: Structural Defects (object and scene structure)\n2.1 Human anatomy \u2014 face defects (eyes, teeth, proportions), hand defects (finger count, grip), body issues (extra/missing limbs, impossible joints).\n2.2 Non-human anatomy \u2014 wrong limb count on animals, distorted animal faces, impossible fur/feather/scale rendering. If no animals or non-human creatures appear, return detected: false with evidence \"no non-human subjects present\".\n2.3 Object structural \u2014 distorted mechanical objects, text rendering failures (gibberish signs), wrong scale relationships, impossible topology.\n2.4 Scene composition \u2014 impossible spatial arrangements, missing expected objects, wrong perspective.\n\n## Domain 3: Temporal-Semantic Violations (across-frame, motion-visible only)\n3.1 Motion artefacts \u2014 jittery stationary objects, non-rigid motion of rigid objects, foot sliding during walking, impossible acceleration.\n3.2 Identity and object drift \u2014 face morphing between frames, clothing pattern changing, colour shifting on same object, object count changing.\n3.3 Continuity errors \u2014 objects appearing/disappearing without cause, sudden lighting shifts within a shot, background motion inconsistent with foreground, loop/repeat motion.\n3.4 Causality and physics violations \u2014 irreversibility violation (spilled liquid returning), conservation of matter violation, gravity/momentum failures, cause-effect mismatch.\n\n# SEVERITY BANDS \u2014 use the full range\n\n- L (Low): artefact present but subtle. A viewer would need to pause or look closely to notice. Example: slight waxiness on cheek skin only visible on a still frame; minor edge softness on hair.\n- M (Medium): artefact clearly present at normal playback speed and noticeable to an attentive viewer, but not the dominant feature of the frame. Example: one finger visibly merged with an adjacent finger; background text partially unreadable; a shadow direction that seems off but not impossible.\n- H (High): artefact is obvious, dominant, and would be immediately visible to a casual viewer. Example: hand with six fingers or fingers melting into each other; a person's face morphing shape mid-shot; an object phasing through a solid surface; text that is complete gibberish across the whole sign.\n\n# CONFIDENCE CALIBRATION (integer 0\u2013100)\n\nUse the full range. Anchors:\n- 0\u201320: I am confident this video is Real. No credible AI markers.\n- 30\u201350: Genuinely uncertain. Some markers either way but nothing decisive.\n- 60\u201380: Probably AI-generated. Multiple clear markers but some ambiguity.\n- 85\u2013100: Almost certainly AI-generated. Multiple severe artefacts or a watermark.\n\nA batch of responses all clustered at 60\u201375 indicates you are hedging. Use the extremes when the evidence warrants.\n\nDo NOT default to M. If the artefact is subtle, use L. If it is dominant and unmistakable, use H. A response where every detected family is \"M\" is almost certainly miscalibrated \u2014 reconsider.\n\n# OUTPUT FORMAT\n\nRespond ONLY with the JSON below. No preamble, no code fences, no trailing commentary.\n\n{\n  \"video_verdict\": \"AI-generated\" | \"Real\" | \"Uncertain\",\n  \"confidence\": <integer 0-100>,\n  \"families\": {\n    \"1.1_texture\":          {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"<observation with timestamp 0:XX and spatial location>\"},\n    \"1.2_boundary\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.3_lighting\":         {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"1.4_watermark\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.1_human_anatomy\":    {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.2_non_human_anatomy\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.3_object_structural\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"2.4_scene_composition\":{\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.1_motion\":           {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.2_identity_drift\":   {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.3_continuity\":       {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"},\n    \"3.4_causality\":        {\"detected\": <bool>, \"severity\": \"L\"|\"M\"|\"H\"|\"none\", \"evidence\": \"...\"}\n  },\n  \"most_diagnostic_artefact\": \"<one sentence naming the single most decisive observation, or 'no strong artefacts observed'>\",\n  \"notes\": \"<one to two sentences of additional observation, or empty string>\"\n}\n\n# RULES\n\n1. Mark \"detected\": true only when you observe specific evidence in the video, not on suspicion.\n2. When \"detected\" is false, set \"severity\": \"none\" and \"evidence\" to a brief reason.\n3. Every \"evidence\" field for a detected artefact must include (a) a timestamp 0:XX and (b) a spatial location (e.g. \"bottom-left\", \"on the subject's right hand\", \"in the mirror reflection\", \"across the whole frame\").\n4. \"Real\" is a valid verdict. Do not assume AI-generation.\n5. Use the full L/M/H range as calibrated above.\n6. Output must be valid JSON, all 12 family keys present, no additional keys, no markdown."
print(f"Prompt: {len(SYSTEM_PROMPT)} chars, ~{len(SYSTEM_PROMPT)//4} tokens")


Prompt: 6378 chars, ~1594 tokens


## 6. Analyser with retry and parse validation

In [6]:
FAMILIES = ["1.1_texture", "1.2_boundary", "1.3_lighting", "1.4_watermark",
            "2.1_human_anatomy", "2.2_non_human_anatomy", "2.3_object_structural",
            "2.4_scene_composition", "3.1_motion", "3.2_identity_drift",
            "3.3_continuity", "3.4_causality"]

def extract_json(text):
    # Return parsed dict or None
    if not text or not isinstance(text, str):
        return None
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    t = t.strip()
    obj = None
    try:
        obj = json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            try: obj = json.loads(m.group(0))
            except Exception: return None
        else:
            return None
    if isinstance(obj, list):
        obj = obj[0] if obj else None
    if not isinstance(obj, dict) or "families" not in obj:
        return None
    return obj

def has_full_schema(obj):
    fams = obj.get("families")
    return isinstance(fams, dict) and all(f in fams for f in FAMILIES)

def call_once(video_path, system_prompt):
    # One API call. Returns (raw_text, usage_dict).
    uploaded = client.files.upload(file=video_path)
    while uploaded.state.name == "PROCESSING":
        time.sleep(2)
        uploaded = client.files.get(name=uploaded.name)
    if uploaded.state.name != "ACTIVE":
        raise RuntimeError(f"Upload failed: {uploaded.state.name}")
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=[uploaded, "Analyse this video per your system instruction."],
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=TEMPERATURE,
                response_mime_type="application/json",
            ),
        )
    finally:
        try: client.files.delete(name=uploaded.name)
        except Exception: pass
    um = resp.usage_metadata
    usage = {
        "input_tokens":  getattr(um, "prompt_token_count", None) if um else None,
        "output_tokens": getattr(um, "candidates_token_count", None) if um else None,
        "thinking_tokens": getattr(um, "thoughts_token_count", None) if um else None,
    }
    return resp.text, usage

def analyse_video_with_retries(video_path, system_prompt):
    # Call, parse, retry up to MAX_RETRIES if parse or schema fails.
    for attempt in range(MAX_RETRIES + 1):
        raw, usage = call_once(video_path, system_prompt)
        obj = extract_json(raw)
        if obj is not None and has_full_schema(obj):
            return {"status": "ok", "attempt": attempt + 1, "parsed": obj,
                    "raw_response": raw, "usage": usage}
        # parse or schema failure - retry
    return {"status": "parse_failed", "attempt": MAX_RETRIES + 1,
            "parsed": None, "raw_response": raw, "usage": usage}


## 7. Main run — checkpointed

If this cell crashes or you close the tab, just rerun it. It will skip anything already logged in `gemini_results.jsonl`.


In [7]:
# Load checkpoint: set of video_ids already processed (any status)
done_ids = set()
if LOG_PATH.exists():
    with LOG_PATH.open() as f:
        for line in f:
            try:
                rec = json.loads(line)
                done_ids.add(rec.get("video_id"))
            except Exception:
                pass
print(f"Checkpoint: {len(done_ids)} videos already logged. Will skip them.")

# Cost tracker — Gemini 3.1 Pro standard tier
PRICE_INPUT_PER_M  = 2.0
PRICE_OUTPUT_PER_M = 12.0
total_in = total_out = total_think = 0
n_ok = n_fail = 0
t_start = time.time()

remaining = [(g, p) for g, p in VIDEO_INDEX if pathlib.Path(p).stem not in done_ids]
print(f"To process this run: {len(remaining)}")

with LOG_PATH.open("a") as f:
    for i, (gen_label, video_path) in enumerate(remaining, start=1):
        vid = pathlib.Path(video_path).stem
        print(f"[{i:3d}/{len(remaining)}] {gen_label:12s} {vid[:50]:50s}", end="  ")
        t0 = time.time()
        try:
            result = analyse_video_with_retries(video_path, SYSTEM_PROMPT)
            record = {
                "video_id": vid,
                "generator": gen_label,
                "prompt_variant": "B_calibrated",
                "model": MODEL,
                "status": result["status"],
                "attempts": result["attempt"],
                "elapsed_s": round(time.time() - t0, 1),
                "usage": result["usage"],
                "parsed": result["parsed"],
                "raw_response": result["raw_response"],
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
            if result["status"] == "ok":
                n_ok += 1
            else:
                n_fail += 1
            # cost accumulation
            u = result["usage"] or {}
            total_in    += u.get("input_tokens") or 0
            total_out   += u.get("output_tokens") or 0
            total_think += u.get("thinking_tokens") or 0
            cost = (total_in / 1e6) * PRICE_INPUT_PER_M + \
                   ((total_out + total_think) / 1e6) * PRICE_OUTPUT_PER_M
            elapsed_min = (time.time() - t_start) / 60
            print(f"{result['status']:12s}  a={result['attempt']}  "
                  f"({record['elapsed_s']}s)  running: ok={n_ok} fail={n_fail}  "
                  f"~${cost:.2f}  {elapsed_min:.1f} min")
        except Exception as e:
            record = {
                "video_id": vid,
                "generator": gen_label,
                "prompt_variant": "B_calibrated",
                "model": MODEL,
                "status": "error",
                "error": str(e)[:400],
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
            n_fail += 1
            print(f"ERROR: {str(e)[:80]}")
        f.write(json.dumps(record) + "\n")
        f.flush()
        time.sleep(SLEEP_S)

print(f"\n=== Run complete ===")
print(f"OK: {n_ok}  Failed: {n_fail}")
print(f"Tokens — input: {total_in:,}  output: {total_out:,}  thinking: {total_think:,}")
cost = (total_in/1e6)*PRICE_INPUT_PER_M + ((total_out+total_think)/1e6)*PRICE_OUTPUT_PER_M
print(f"Estimated cost this run: ${cost:.2f}")


Checkpoint: 191 videos already logged. Will skip them.
To process this run: 89
[  1/89] LTX          w2_020_ltx_20260719_110441                          ok            a=1  (46.8s)  running: ok=1 fail=0  ~$0.03  0.8 min
[  2/89] LTX          w2_038_ltx_20260719_111311                          ok            a=1  (96.4s)  running: ok=2 fail=0  ~$0.08  2.4 min
[  3/89] Hunyuan      w2_029_hunyuan_20260719_145312                      ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
[  4/89] Hunyuan      w2_031_hunyuan_20260719_145855                      ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
[  5/89] Wan          w2_015_wan_20260719_172211                          ok            a=1  (78.0s)  running: ok=3 fail=2  ~$0.11  4.1 min
[  6/89] Wan          w2_033_wan_20260719_183755                          ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
[  7/89] Seedan

## 8. Convert JSONL → CSV in master-spreadsheet column layout

Emits `gemini_results.csv`. Column names match the Data sheet of `MLLM_Analysis_Log.xlsx` exactly — paste straight in.


In [8]:
BASE_COLS = ["video_id", "generator", "mllm", "batch_date", "verdict", "confidence"]
FAM_COLS = []
for key in FAMILIES:
    FAM_COLS += [f"{key}__detected", f"{key}__severity", f"{key}__evidence"]
TAIL_COLS = ["most_diagnostic_artefact", "notes", "review_flag"]
ALL_COLS = BASE_COLS + FAM_COLS + TAIL_COLS

records = []
with LOG_PATH.open() as f:
    for line in f:
        try: records.append(json.loads(line))
        except Exception: pass

rows = []
for rec in records:
    parsed = rec.get("parsed") or {}
    fams = parsed.get("families", {}) if isinstance(parsed, dict) else {}
    row = {
        "video_id": rec.get("video_id"),
        "generator": rec.get("generator"),
        "mllm": "Gemini_3.1_Pro",
        "batch_date": (rec.get("timestamp") or "")[:10],
        "verdict": parsed.get("video_verdict", ""),
        "confidence": parsed.get("confidence", ""),
        "most_diagnostic_artefact": parsed.get("most_diagnostic_artefact", ""),
        "notes": parsed.get("notes", ""),
        "review_flag": "" if rec.get("status") == "ok" else f"STATUS: {rec.get('status')}",
    }
    for key in FAMILIES:
        f = fams.get(key) if isinstance(fams, dict) else None
        if isinstance(f, dict):
            det = f.get("detected")
            row[f"{key}__detected"] = "TRUE" if det is True else ("FALSE" if det is False else "")
            row[f"{key}__severity"] = f.get("severity", "")
            row[f"{key}__evidence"] = f.get("evidence", "")
        else:
            row[f"{key}__detected"] = ""
            row[f"{key}__severity"] = ""
            row[f"{key}__evidence"] = ""
    rows.append(row)

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=ALL_COLS)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print(f"Wrote {len(rows)} rows to {CSV_PATH}")
print(f"\nTo import: open MLLM_Analysis_Log.xlsx -> Data sheet -> click A3 -> Paste Special -> values")


Wrote 280 rows to /content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.csv

To import: open MLLM_Analysis_Log.xlsx -> Data sheet -> click A3 -> Paste Special -> values


## 9. Quick summary — what got done, what needs re-running

In [9]:
by_status_by_gen = {}
for rec in records:
    key = (rec.get("generator", "?"), rec.get("status", "?"))
    by_status_by_gen[key] = by_status_by_gen.get(key, 0) + 1

gens = sorted({g for g, _ in by_status_by_gen})
statuses = sorted({s for _, s in by_status_by_gen})
print(f"{'Generator':14s}  " + "  ".join(f"{s:>14s}" for s in statuses) + "   total   expected")
for g in gens:
    counts = {s: by_status_by_gen.get((g, s), 0) for s in statuses}
    total = sum(counts.values())
    exp = EXPECTED_COUNTS.get(g, "?")
    print(f"{g:14s}  " + "  ".join(f"{counts[s]:>14d}" for s in statuses) + f"   {total:>5d}   {exp}")

failed = [r for r in records if r.get("status") != "ok"]
if failed:
    print(f"\n{len(failed)} records need attention:")
    for r in failed[:10]:
        print(f"  {r.get('generator'):12s} {r.get('video_id')[:60]:60s} {r.get('status')}")
    if len(failed) > 10:
        print(f"  ... and {len(failed)-10} more")

# If you want to retry only the failures, delete the failed lines from
# gemini_results.jsonl and rerun Cell 7 - checkpoint logic will pick them up.


Generator                error              ok    parse_failed   total   expected
Gemini_Omni                  0              32               0      32   32
Hunyuan                      2              38               0      40   40
Kling                        0              32               0      32   32
LTX                          0              40               0      40   40
Pexels                      15              48               1      64   64
Seedance                     2              30               0      32   32
Wan                          1              39               0      40   40

21 records need attention:
  Hunyuan      w2_029_hunyuan_20260719_145312                               error
  Hunyuan      w2_031_hunyuan_20260719_145855                               error
  Wan          w2_033_wan_20260719_183755                                   error
  Seedance     w2_019_seedance                                              error
  Seedance     w2_022_seedance